# LLM Error Analysis

This notebook analyzes errors from the downstream LLM-assisted MITRE ATT&CK mapping experiment. The goal is to determine whether incorrect final predictions were primarily caused by the retrieval stage or by the LLM's candidate selection.

For each retrieval method—Bag-of-Words, TF-IDF, and Sentence Embeddings—errors are separated into two main categories:

- **Retrieval Failure:** The ground-truth ATT&CK technique was not included in the retrieved Top-5 candidates, meaning the LLM could not select the correct answer.
- **LLM Selection Failure:** The ground-truth technique was successfully retrieved within the Top 5, but the LLM selected a different candidate.

Successful predictions are also recorded to compare the relative contribution of retrieval and LLM selection to overall pipeline performance.

This analysis helps identify the primary bottleneck in the retrieval-augmented ATT&CK mapping pipeline.

In [2]:
#load
import pandas as pd

bow = pd.read_pickle(
    "bow_llm_results.pkl"
)

tfidf = pd.read_pickle(
    "tfidf_llm_results.pkl"
)

embedding = pd.read_pickle(
    "embedding_llm_results.pkl"
)

- Success — retrieval found the correct technique and Gemini selected it.
- Retrieval failure — correct technique was not in Top 5.
- LLM selection failure — correct technique was in Top 5, but Gemini selected something else.

In [3]:
def classify_outcome(row, retrieval_column):

    if not row[retrieval_column]:
        return "Retrieval Failure"

    elif row["llm_correct"]:
        return "Success"

    else:
        return "LLM Selection Failure"

In [4]:
bow["outcome"] = bow.apply(
    lambda row: classify_outcome(
        row,
        "bow_retrieval_success"
    ),
    axis=1
)

tfidf["outcome"] = tfidf.apply(
    lambda row: classify_outcome(
        row,
        "tfidf_retrieval_success"
    ),
    axis=1
)

embedding["outcome"] = embedding.apply(
    lambda row: classify_outcome(
        row,
        "embedding_retrieval_success"
    ),
    axis=1
)

In [5]:
print("Bag-of-Words")
print(bow["outcome"].value_counts())

print("\nTF-IDF")
print(tfidf["outcome"].value_counts())

print("\nSentence Embeddings")
print(embedding["outcome"].value_counts())

Bag-of-Words
outcome
Retrieval Failure        15
Success                   4
LLM Selection Failure     1
Name: count, dtype: int64

TF-IDF
outcome
Retrieval Failure    16
Success               4
Name: count, dtype: int64

Sentence Embeddings
outcome
Retrieval Failure        10
Success                   7
LLM Selection Failure     3
Name: count, dtype: int64


In [6]:
#summary table
error_summary = pd.DataFrame({
    "Pipeline": [
        "Bag-of-Words + Gemini",
        "TF-IDF + Gemini",
        "Sentence Embeddings + Gemini"
    ],

    "Success": [
        (bow["outcome"] == "Success").sum(),
        (tfidf["outcome"] == "Success").sum(),
        (embedding["outcome"] == "Success").sum()
    ],

    "Retrieval Failure": [
        (bow["outcome"] == "Retrieval Failure").sum(),
        (tfidf["outcome"] == "Retrieval Failure").sum(),
        (embedding["outcome"] == "Retrieval Failure").sum()
    ],

    "LLM Selection Failure": [
        (bow["outcome"] == "LLM Selection Failure").sum(),
        (tfidf["outcome"] == "LLM Selection Failure").sum(),
        (embedding["outcome"] == "LLM Selection Failure").sum()
    ]
})

error_summary

,Pipeline,Success,Retrieval Failure,LLM Selection Failure
0,Bag-of-Words + Gemini,4,15,1
1,TF-IDF + Gemini,4,16,0
2,Sentence Embeddings + Gemini,7,10,3


In [7]:
#save
error_summary.to_csv(
    "../results/llm_error_summary.csv",
    index=False
)

print("LLM error summary saved.")

LLM error summary saved.


Error decomposition showed that retrieval was the primary bottleneck in the LLM-assisted mapping pipeline. Across the three retrieval conditions, 41 of 45 incorrect predictions (91.1%) occurred because the ground-truth ATT&CK technique was absent from the retrieved Top-5 candidates. Only four errors occurred when the correct technique was available but Gemini selected another candidate. This suggests that, within this evaluation sample, improving candidate retrieval may have a greater effect on end-to-end ATT&CK mapping performance than improving LLM candidate selection.

**next, inspect LLM failures**

In [8]:
bow_llm_failures = bow[
    bow["outcome"] == "LLM Selection Failure"
].copy()

embedding_llm_failures = embedding[
    embedding["outcome"] == "LLM Selection Failure"
].copy()

In [9]:
#bow failure
bow_llm_failures[
    [
        "sentence",
        "technique_id",
        "llm_prediction",
        "bow_retrieval_success"
    ]
]

,sentence,technique_id,llm_prediction,bow_retrieval_success
2965,These attempts are primarily performed with an...,T1110,T1588.006,True


In [10]:
#3 embedding failure
embedding_llm_failures[
    [
        "sentence",
        "technique_id",
        "llm_prediction",
        "embedding_retrieval_success"
    ]
]

,sentence,technique_id,llm_prediction,embedding_retrieval_success
2719,Before attempting to overwrite as much data as...,T1070.004,T1485,True
2965,These attempts are primarily performed with an...,T1110,T1595.002,True
2742,TA0010 Exfiltration T1020 Automated Exfiltrati...,T1041,T1020,True


### Qualitative LLM Error Analysis

Only four LLM selection failures occurred across the three retrieval conditions: one with Bag-of-Words and three with sentence embeddings. Manual inspection suggests that some errors involved confusion between related ATT&CK behaviors. For example, the LLM selected T1020 instead of the ground-truth T1041 in an exfiltration-related example, indicating difficulty distinguishing between closely related techniques.

The same T1110-labeled threat description also produced incorrect LLM selections under both Bag-of-Words and embedding retrieval, despite the correct technique being available in the Top-5 candidates. This suggests that some errors may result from ambiguous or competing behavioral signals within the threat description rather than retrieval failure alone.

Overall, LLM selection errors were relatively uncommon compared with retrieval failures. The primary limitation of the pipeline was therefore retrieving the correct ATT&CK technique into the candidate set rather than selecting it once available.